In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
head_size = 32
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)

In [4]:
# module declaration
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len):
        super().__init__()

        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.position_embedding = nn.Embedding(num_embeddings=sequence_len, embedding_dim=embedding_size)

    def forward(self, x):
        return self.token_embedding(x) + self.position_embedding(torch.arange(x.size(1)))

class SelfAttention(nn.Module):
    def __init__(self, embedding_size, head_size, sequence_len):
        super().__init__()

        self.head_size = head_size
        self.query = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.value = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)

        tril = torch.tril(torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, x):
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)

        T = x.size(1)

        scores = query @ key.transpose(dim0=-2, dim1=-1) * self.head_size ** -0.5
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)

        return weights @ value

In [5]:
# model declaration
class Model(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len, head_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            TokenAndPositionEmbedding(vocab_size=vocab_size, embedding_size=embedding_size, sequence_len=sequence_len),
            SelfAttention(embedding_size=embedding_size, head_size=head_size, sequence_len=sequence_len),
            nn.Linear(in_features=head_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=vocab_size),
        )

    def forward(self, x):
        return self.layers(x)

In [6]:
# model initialization
model = Model(vocab_size=vocab_size, embedding_size=embedding_size, sequence_len=sequence_len, head_size=head_size, hidden_size=hidden_size)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 32353


In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits = model(x_batch)
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.155622482299805, val loss = 4.157438278198242


Step 1000: train loss = 2.3391404151916504, val loss = 2.348768711090088


Step 2000: train loss = 1.966465950012207, val loss = 2.1273019313812256


Step 3000: train loss = 1.9576096534729004, val loss = 2.0633180141448975


Step 4000: train loss = 1.8717401027679443, val loss = 2.016059637069702


Step 4999: train loss = 1.8200492858886719, val loss = 2.01572585105896


In [8]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits = model(i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long)

In [9]:
print(''.join(output))

ROMEO:
Aus and have notunling th of Say; brother'd likl anceers! that the may becray and them,
Whilly h wassion tood they
Best hings,
Belok,
Wheper Caim, your bestemes wall sunram; I prace, my be ge,
The thenchor mean hathin that the duklessenpint that sard flune tor to withed thou whe 'ting.


AUFIO:
By onto dughe hreep
pirance;
Aw preceiven andeman:
By mis, Thentand's thous sir;
No, ting hir; and to't, neet, I pose quit, wh them with half, whong him weak thou hown:
O, Topett: Poff
ive my liferiever's not devily
Thenceer.
And gold live dother h! Whely vove I uppergainge,
Way ford, I hop and h thter'd, heare thou so.

AURY:
Eve father trie had the wnow netwast:
They exfor the when in Stithate
Ar, deathis is wither sun to stroul pe as as wre hee foue then of my spulestay
sow artain.

VOLUMNIA:
ES MONESCALUS:
So han!
OMEON:
I slatiner't's not my to hnow the whiched lachter ing mak you h deathinualteer wireased up nown:
Nopreved land borese pey good I hears ranching you hind deard me word